In [ ]:
import requests
import pandas as pd
import time
import base64
import random
from datetime import datetime
from collections import Counter

In [ ]:
random.seed(42)

In [ ]:
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"   # <-- paste your token here
OUTPUT_FILE  = "github_projects_1000.xlsx"
DELAY        = 1.5  # seconds between requests

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

In [ ]:
SEARCH_QUERIES = [

    # ── Ecommerce (~120 target) ──────────────────────────────────────────────
    ("topic:ecommerce stars:>50 language:javascript",        "Ecommerce"),
    ("topic:ecommerce stars:>50 language:python",            "Ecommerce"),
    ("topic:ecommerce stars:>30 language:php",               "Ecommerce"),
    ("topic:ecommerce stars:>30 language:java",              "Ecommerce"),
    ("topic:shopping-cart stars:>30",                        "Ecommerce"),
    ("ecommerce platform fullstack stars:>100",              "Ecommerce"),
    ("topic:woocommerce stars:>40",                          "Ecommerce"),
    ("online marketplace fullstack stars:>50",               "Ecommerce"),
    ("topic:shopify stars:>30",                              "Ecommerce"),
    ("multi vendor ecommerce stars:>30",                     "Ecommerce"),

    # ── Healthcare (~100 target) ─────────────────────────────────────────────
    ("topic:healthcare stars:>30",                           "Healthcare"),
    ("hospital management system stars:>20",                 "Healthcare"),
    ("topic:medical-records stars:>20",                      "Healthcare"),
    ("topic:telemedicine stars:>20",                         "Healthcare"),
    ("patient management system stars:>20",                  "Healthcare"),
    ("topic:ehr stars:>20",                                  "Healthcare"),
    ("clinic management system stars:>15",                   "Healthcare"),
    ("healthcare app fullstack stars:>20",                   "Healthcare"),
    ("pharmacy management system stars:>15",                 "Healthcare"),
    ("topic:health-app stars:>20",                           "Healthcare"),

    # ── Education (~100 target) ──────────────────────────────────────────────
    ("topic:lms stars:>50",                                  "Education"),
    ("topic:elearning stars:>30",                            "Education"),
    ("online learning management system stars:>50",          "Education"),
    ("topic:online-course stars:>30",                        "Education"),
    ("topic:moodle stars:>30",                               "Education"),
    ("student management system stars:>20",                  "Education"),
    ("topic:quiz-app stars:>30",                             "Education"),
    ("virtual classroom stars:>20",                          "Education"),
    ("school management system stars:>20",                   "Education"),
    ("education platform fullstack stars:>30",               "Education"),

    # ── Finance (~100 target) ────────────────────────────────────────────────
    ("topic:fintech stars:>50",                              "Finance"),
    ("personal finance tracker stars:>30",                   "Finance"),
    ("topic:banking stars:>30",                              "Finance"),
    ("topic:cryptocurrency stars:>50",                       "Finance"),
    ("budget tracker app stars:>30",                         "Finance"),
    ("topic:payment-gateway stars:>30",                      "Finance"),
    ("investment portfolio tracker stars:>30",               "Finance"),
    ("loan management system stars:>20",                     "Finance"),
    ("accounting software stars:>30",                        "Finance"),
    ("topic:expense-tracker stars:>30",                      "Finance"),

    # ── Social Media (~90 target) ────────────────────────────────────────────
    ("topic:social-network stars:>50",                       "Social Media"),
    ("social media platform fullstack stars:>100",           "Social Media"),
    ("topic:chat-app stars:>50",                             "Social Media"),
    ("topic:messaging-app stars:>40",                        "Social Media"),
    ("real time chat application stars:>50",                 "Social Media"),
    ("topic:twitter-clone stars:>30",                        "Social Media"),
    ("topic:instagram-clone stars:>30",                      "Social Media"),
    ("community forum platform stars:>30",                   "Social Media"),
    ("topic:social-app stars:>30",                           "Social Media"),

    # ── Logistics (~90 target) ───────────────────────────────────────────────
    ("topic:logistics stars:>30",                            "Logistics"),
    ("fleet management system stars:>20",                    "Logistics"),
    ("delivery tracking app stars:>30",                      "Logistics"),
    ("warehouse management system stars:>20",                "Logistics"),
    ("supply chain management stars:>20",                    "Logistics"),
    ("topic:tracking-system stars:>20",                      "Logistics"),
    ("route optimization system stars:>20",                  "Logistics"),
    ("inventory management system stars:>30",                "Logistics"),
    ("shipping tracking system stars:>20",                   "Logistics"),

    # ── Real Estate (~90 target) ─────────────────────────────────────────────
    ("topic:real-estate stars:>30",                          "Real Estate"),
    ("property listing platform stars:>20",                  "Real Estate"),
    ("real estate management system stars:>20",              "Real Estate"),
    ("topic:property-management stars:>20",                  "Real Estate"),
    ("rental management system stars:>20",                   "Real Estate"),
    ("real estate crm stars:>15",                            "Real Estate"),
    ("property booking system stars:>15",                    "Real Estate"),
    ("house rental platform stars:>20",                      "Real Estate"),
    ("real estate website fullstack stars:>20",              "Real Estate"),

    # ── HR & Recruitment (~90 target) ────────────────────────────────────────
    ("topic:hrms stars:>20",                                 "HR & Recruitment"),
    ("applicant tracking system stars:>30",                  "HR & Recruitment"),
    ("topic:payroll stars:>20",                              "HR & Recruitment"),
    ("human resource management stars:>20",                  "HR & Recruitment"),
    ("employee management system stars:>20",                 "HR & Recruitment"),
    ("topic:recruitment stars:>20",                          "HR & Recruitment"),
    ("leave management system stars:>15",                    "HR & Recruitment"),
    ("performance review system stars:>15",                  "HR & Recruitment"),
    ("job portal fullstack stars:>20",                       "HR & Recruitment"),

    # ── IoT & Smart Systems (~80 target) ─────────────────────────────────────
    ("topic:iot-dashboard stars:>30",                        "IoT & Smart Systems"),
    ("smart home automation stars:>30",                      "IoT & Smart Systems"),
    ("topic:iot stars:>50 language:python",                  "IoT & Smart Systems"),
    ("topic:smart-home stars:>30",                           "IoT & Smart Systems"),
    ("iot monitoring system stars:>20",                      "IoT & Smart Systems"),
    ("topic:home-automation stars:>30",                      "IoT & Smart Systems"),
    ("sensor data dashboard stars:>20",                      "IoT & Smart Systems"),
    ("topic:iot-platform stars:>20",                         "IoT & Smart Systems"),

    # ── Travel & Hospitality (~80 target) ────────────────────────────────────
    ("hotel booking system stars:>30",                       "Travel & Hospitality"),
    ("travel app fullstack stars:>30",                       "Travel & Hospitality"),
    ("topic:hotel-management stars:>20",                     "Travel & Hospitality"),
    ("flight booking system stars:>20",                      "Travel & Hospitality"),
    ("restaurant management system stars:>20",               "Travel & Hospitality"),
    ("topic:booking-system stars:>30",                       "Travel & Hospitality"),
    ("travel booking platform stars:>20",                    "Travel & Hospitality"),
    ("tour management system stars:>15",                     "Travel & Hospitality"),
]

In [ ]:
FRONTEND_KEYWORDS = {
    "React":        ["react", "reactjs", "react.js", "jsx", "create-react-app",
                     "redux", "react-router", "react-query", "react-hooks"],
    "Next.js":      ["next.js", "nextjs", "next js", "vercel/next"],
    "Vue.js":       ["vue", "vuejs", "vue.js", "nuxtjs", "nuxt", "vuex", "vite+vue"],
    "Angular":      ["angular", "angularjs", "ng-", "angular cli", "rxjs", "ngrx", "@angular"],
    "Svelte":       ["svelte", "sveltekit"],
    "jQuery":       ["jquery"],
    "Flutter":      ["flutter", "dart"],
    "React Native": ["react native", "expo", "react-native"],
    "Bootstrap":    ["bootstrap", "tailwind"],
    "HTML/CSS":     ["html", "css3", "static site"],
}

BACKEND_KEYWORDS = {
    "Node.js":     ["node.js", "nodejs", "node js", "express", "expressjs",
                    "nestjs", "nest.js", "fastify", "koa", "hapi", "socket.io"],
    "Django":      ["django", "django rest", "drf", "django-rest-framework", "django admin"],
    "Flask":       ["flask", "flask-restful", "flask api"],
    "FastAPI":     ["fastapi", "fast api", "uvicorn", "starlette"],
    "Laravel":     ["laravel", "artisan", "eloquent", "blade"],
    "Spring Boot": ["spring boot", "springboot", "spring-boot", "spring mvc",
                    "java spring", "spring framework"],
    "Rails":       ["ruby on rails", "rails", "ror", "sinatra"],
    "Go":          ["golang", "gin", "go fiber", "echo framework", "gorilla"],
    "ASP.NET":     ["asp.net", ".net core", "dotnet", "c# api", "blazor"],
    "PHP":         ["php", "codeigniter", "symfony"],
}

DATABASE_KEYWORDS = {
    "MongoDB":       ["mongodb", "mongoose", "mongo db", "atlas", "nosql"],
    "PostgreSQL":    ["postgresql", "postgres", "psql", "pg ", "supabase",
                      "neon", "timescaledb", "pgadmin"],
    "MySQL":         ["mysql", "mariadb", "phpmyadmin"],
    "SQLite":        ["sqlite", "sqlite3"],
    "Firebase":      ["firebase", "firestore", "realtime database", "firebase auth"],
    "Redis":         ["redis", "redis cache", "bull queue"],
    "Supabase":      ["supabase"],
    "DynamoDB":      ["dynamodb", "dynamo db", "aws dynamodb"],
    "Cassandra":     ["cassandra", "apache cassandra"],
    "Elasticsearch": ["elasticsearch", "elastic search", "opensearch"],
    "SQL Server":    ["sql server", "mssql", "microsoft sql"],
    "Oracle":        ["oracle db", "oracle database"],
}

LANGUAGE_TO_BACKEND = {
    "JavaScript": "Node.js",  "TypeScript": "Node.js",
    "Python":     "Django",   "PHP":        "Laravel",
    "Java":       "Spring Boot","Kotlin":   "Spring Boot",
    "Ruby":       "Rails",    "Go":         "Go",
    "C#":         "ASP.NET",  "Dart":       "Flutter",
}
LANGUAGE_TO_FRONTEND = {
    "JavaScript": "React", "TypeScript": "React", "Dart": "Flutter",
}

In [ ]:
# ── DOMAIN DEFAULTS (fallback when detection fails) ───────────────────────────
DOMAIN_DEFAULTS = {
    "Ecommerce":           ("React",   "Node.js",     "MongoDB"),
    "Healthcare":          ("React",   "Django",      "PostgreSQL"),
    "Education":           ("React",   "Node.js",     "MongoDB"),
    "Finance":             ("Angular", "Spring Boot", "PostgreSQL"),
    "Social Media":        ("React",   "Node.js",     "MongoDB"),
    "Logistics":           ("React",   "Node.js",     "PostgreSQL"),
    "Real Estate":         ("React",   "Django",      "PostgreSQL"),
    "HR & Recruitment":    ("Angular", "Spring Boot", "PostgreSQL"),
    "IoT & Smart Systems": ("React",   "Node.js",     "MongoDB"),
    "Travel & Hospitality":("React",   "Node.js",     "PostgreSQL"),
}


In [ ]:
# ── NFR RULES per domain ──────────────────────────────────────────────────────
DOMAIN_NFRS = {
    "Ecommerce":           ["scalability","security","high availability",
                            "performance","SEO optimization","mobile responsiveness",
                            "PCI-DSS compliance","load balancing"],
    "Healthcare":          ["HIPAA compliance","data privacy","high availability",
                            "security","audit logging","reliability",
                            "data encryption","access control"],
    "Education":           ["scalability","performance","accessibility",
                            "mobile responsiveness","real-time support",
                            "uptime","WCAG compliance","multi-device support"],
    "Finance":             ["security","PCI-DSS compliance","high availability",
                            "audit trail","performance","encryption",
                            "regulatory compliance","data integrity"],
    "Social Media":        ["scalability","real-time performance","low latency",
                            "CDN integration","high availability",
                            "media optimization","abuse prevention"],
    "Logistics":           ["real-time updates","GPS integration","reliability",
                            "scalability","offline support","performance","data accuracy"],
    "Real Estate":         ["SEO","performance","mobile responsiveness",
                            "security","scalability","high availability","fast image loading"],
    "HR & Recruitment":    ["data privacy","security","scalability","compliance",
                            "usability","reliability","role-based access","audit logging"],
    "IoT & Smart Systems": ["real-time processing","scalability","reliability",
                            "low latency","security","edge computing support",
                            "data retention","interoperability"],
    "Travel & Hospitality":["performance","scalability","high availability",
                            "mobile responsiveness","SEO","security",
                            "fast search response","multi-language support"],
}

In [ ]:
# ── SIZE / BUDGET / DURATION RULES ───────────────────────────────────────────
SIZE_CONFIG = {
    "Small":  {"team": (1, 4),   "budget": "Low",    "duration": (1, 5)},
    "Medium": {"team": (4, 12),  "budget": "Medium", "duration": (4, 12)},
    "Large":  {"team": (12, 40), "budget": "High",   "duration": (8, 24)},
}

In [ ]:
# ── HELPER FUNCTIONS ─────────────────────────────────────────────────────────
def detect_tech(text, keyword_map, default="Unknown"):
    tl = text.lower()
    for tech, keywords in keyword_map.items():
        if any(kw in tl for kw in keywords):
            return tech
    return default

def get_readme(owner, repo):
    url = f"https://api.github.com/repos/{owner}/{repo}/readme"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            content = r.json().get("content", "")
            return base64.b64decode(content).decode("utf-8", errors="ignore")
    except Exception:
        pass
    return ""

def get_repo_languages(owner, repo):
    url = f"https://api.github.com/repos/{owner}/{repo}/languages"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return {}

def classify_size(stars, forks):
    score = stars + forks * 2
    if score < 100:   return "Small"
    elif score < 600: return "Medium"
    else:             return "Large"

def fill_missing_columns(domain, size_label):
    """Generate NFR, team size, budget, duration from domain + size rules."""
    nfrs = DOMAIN_NFRS.get(domain, ["scalability","security","performance","reliability"])
    selected_nfrs = ", ".join(random.sample(nfrs, min(3, len(nfrs))))

    cfg      = SIZE_CONFIG.get(size_label, SIZE_CONFIG["Medium"])
    team     = random.randint(*cfg["team"])
    budget   = cfg["budget"]
    duration = random.randint(*cfg["duration"])

    return selected_nfrs, team, budget, duration

def check_rate_limit():
    """Check remaining API calls and wait if needed."""
    r = requests.get("https://api.github.com/rate_limit", headers=HEADERS, timeout=10)
    if r.status_code == 200:
        data      = r.json()
        remaining = data["resources"]["search"]["remaining"]
        reset_at  = data["resources"]["search"]["reset"]
        if remaining < 3:
            wait = max(reset_at - time.time() + 5, 0)
            print(f"  ⚠ Rate limit low ({remaining} left). Waiting {int(wait)}s...")
            time.sleep(wait)
        return remaining
    return 10

In [ ]:
# ── MAIN COLLECTOR ────────────────────────────────────────────────────────────
def search_repos(query, domain, per_page=30):
    url    = "https://api.github.com/search/repositories"
    params = {"q": query, "sort": "stars", "order": "desc",
              "per_page": per_page, "page": 1}
    try:
        r = requests.get(url, headers=HEADERS, params=params, timeout=15)
        if r.status_code == 403:
            print("  ⚠ Rate limit hit — waiting 60s...")
            time.sleep(62)
            r = requests.get(url, headers=HEADERS, params=params, timeout=15)
        if r.status_code != 200:
            print(f"  ✗ HTTP {r.status_code}")
            return []
    except Exception as e:
        print(f"  ✗ {e}")
        return []

    records = []
    for item in r.json().get("items", []):
        owner    = item["owner"]["login"]
        repo     = item["name"]
        stars    = item.get("stargazers_count", 0)
        forks    = item.get("forks_count", 0)
        language = item.get("language") or "Unknown"
        topics   = item.get("topics", [])
        desc     = item.get("description") or ""
        html_url = item.get("html_url", "")

        print(f"    → {owner}/{repo} ({stars}★)")

        # Fetch README for tech detection
        readme    = get_readme(owner, repo)
        full_text = f"{desc} {' '.join(topics)} {readme[:4000]}"

        # ── Detect frontend ──────────────────────────────────────────────
        frontend = detect_tech(full_text, FRONTEND_KEYWORDS)

        # ── Detect backend ───────────────────────────────────────────────
        backend = detect_tech(full_text, BACKEND_KEYWORDS)

        # ── Detect database ──────────────────────────────────────────────
        database = detect_tech(full_text, DATABASE_KEYWORDS)

        # ── Language API fallback (only if still Unknown) ────────────────
        if backend == "Unknown" or frontend == "Unknown":
            langs = get_repo_languages(owner, repo)
            time.sleep(0.3)
            if langs:
                primary = max(langs, key=langs.get)
                if backend == "Unknown" and primary in LANGUAGE_TO_BACKEND:
                    backend = LANGUAGE_TO_BACKEND[primary]
                if frontend == "Unknown":
                    for lang in langs:
                        if lang in LANGUAGE_TO_FRONTEND:
                            frontend = LANGUAGE_TO_FRONTEND[lang]
                            break

        # ── Domain default fallback ──────────────────────────────────────
        def_fe, def_be, def_db = DOMAIN_DEFAULTS.get(domain, ("React","Node.js","MongoDB"))
        if frontend == "Unknown": frontend = def_fe
        if backend  == "Unknown": backend  = def_be
        if database == "Unknown": database = def_db

        # ── Deployment detection ─────────────────────────────────────────
        cloud_kws = ["docker","kubernetes","heroku","aws","azure","gcp",
                     "vercel","netlify","digitalocean","render","railway"]
        deployment = "Cloud" if any(kw in full_text.lower() for kw in cloud_kws) else "On-premise"

        # ── Size classification ──────────────────────────────────────────
        size_label = classify_size(stars, forks)

        # ── Fill missing columns ─────────────────────────────────────────
        nfr, team, budget, duration = fill_missing_columns(domain, size_label)

        # ── Functional requirements from topics ──────────────────────────
        funcs = ", ".join(topics[:6]) if topics else "See README"

        records.append({
            "Project_ID":                  "",
            "Project_Name":                repo.replace("-"," ").replace("_"," ").title(),
            "Domain":                      domain,
            "Project_Description":         desc[:250] if desc else "No description",
            "Functional_Requirements":     funcs,
            "Non_Functional_Requirements": nfr,
            "Project_Size":                size_label,
            "Team_Size":                   team,
            "Budget_Level":                budget,
            "Duration_Months":             duration,
            "Deployment":                  deployment,
            "Frontend_Tech":               frontend,
            "Backend_Tech":                backend,
            "Database":                    database,
            "Source":                      "GitHub",
            "GitHub_URL":                  html_url,
            "Stars":                       stars,
            "Forks":                       forks,
            "Primary_Language":            language,
        })

        time.sleep(DELAY)

    return records

In [ ]:
# ── RUN ───────────────────────────────────────────────────────────────────────
def main():
    print("=" * 65)
    print("  GitHub Tech Stack Collector  —  Target: 1000 projects")
    print(f"  Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 65)

    all_records = []
    seen_urls   = set()
    domain_counts = Counter()

    for qi, (query, domain) in enumerate(SEARCH_QUERIES, 1):
        print(f"\n[{qi}/{len(SEARCH_QUERIES)}] [{domain}] {query}")

        # Check rate limit every 10 queries
        if qi % 10 == 0:
            remaining = check_rate_limit()
            print(f"  API calls remaining: {remaining}")

        records = search_repos(query, domain, per_page=30)

        added = 0
        for rec in records:
            if rec["GitHub_URL"] not in seen_urls:
                seen_urls.add(rec["GitHub_URL"])
                all_records.append(rec)
                domain_counts[domain] += 1
                added += 1

        print(f"  ✓ +{added} new | Total unique: {len(all_records)}")
        time.sleep(3)  # pause between queries

        if len(all_records) >= 1100:
            print("\n  🎯 Reached 1100 — stopping early to keep ~1000 after dedup")
            break

    # ── Assign Project IDs ────────────────────────────────────────────────────
    random.shuffle(all_records)
    for i, rec in enumerate(all_records, 1):
        rec["Project_ID"] = f"GH{i:04d}"

    # ── Save to Excel ─────────────────────────────────────────────────────────
    df = pd.DataFrame(all_records)
    df.to_excel(OUTPUT_FILE, index=False, sheet_name="GitHub_Projects")

    # ── Final report ──────────────────────────────────────────────────────────
    print(f"\n{'=' * 65}")
    print(f"  ✓ Saved {len(df)} projects → {OUTPUT_FILE}")
    print(f"\n  Domain breakdown:")
    for domain, count in df["Domain"].value_counts().items():
        print(f"    {domain:<28} {count}")
    print(f"\n  Column completeness:")
    for col in ["Non_Functional_Requirements","Team_Size","Budget_Level","Duration_Months","Frontend_Tech","Backend_Tech","Database"]:
        unknowns = (df[col].astype(str).isin(["Unknown","","nan"])).sum()
        filled   = len(df) - unknowns
        print(f"    {col:<35} {filled}/{len(df)} filled ({100*filled//len(df)}%)")
    print(f"\n  Tech distribution:")
    print(f"    Frontend: {dict(df['Frontend_Tech'].value_counts().head(5))}")
    print(f"    Backend : {dict(df['Backend_Tech'].value_counts().head(5))}")
    print(f"    Database: {dict(df['Database'].value_counts().head(5))}")
    print("=" * 65)

    # ── Auto-download in Colab ─────────────────────────────────────────────
    try:
        from google.colab import files
        files.download(OUTPUT_FILE)
        print("  ⬇ Download started!")
    except ImportError:
        print(f"  File saved locally: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

  GitHub Tech Stack Collector  —  Target: 1000 projects
  Started : 2026-04-25 04:42:35

[1/92] [Ecommerce] topic:ecommerce stars:>50 language:javascript
    → reactioncommerce/reaction (12409★)
    → idurar/idurar-erp-crm (8361★)
    → aimeos/aimeos (5344★)
    → cporter202/API-mega-list (4159★)
    → aimeos/aimeos-headless (2498★)
    → justdjango/django-ecommerce (2329★)
    → adrianhajdin/ecommerce_sanity_stripe (2309★)
    → jamstack-cms/jamstack-ecommerce (1959★)
    → adrianhajdin/project_e_commerce (1889★)
    → ndimatteo/HULL (1443★)
    → jgudo/ecommerce-react (1342★)
    → chec/commercejs-nextjs-demo-store (1071★)
    → MONEI/Shopify-api-node (977★)
    → btahir/next-shopify-starter (902★)
    → storefront-foundation/react-storefront (834★)
    → store-craft/storecraft (791★)
    → GreatStackDev/gocart (756★)
    → ethanselzer/react-image-magnify (663★)
    → reactioncommerce/example-storefront (614★)
    → shamahoque/mern-marketplace (588★)
    → chrisdiana/simplestore (503

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ⬇ Download started!
